In [7]:
from pathlib import Path

REAL_DIR = Path("../data/raw/commonvoice")

print("Current directory:")
print(Path.cwd())

print("\nREAL_DIR:")
print(REAL_DIR.resolve())

print("\nDirectory exists:")
print(REAL_DIR.exists())

print("\nMP3 files:")

mp3_files = list(REAL_DIR.rglob("*.mp3"))

print("Count:", len(mp3_files))

for f in mp3_files[:20]:
    print(f)

Current directory:
/Users/melaniequ/Voice Cloning and Fake Audio Detection/voice_cloning

REAL_DIR:
/Users/melaniequ/Voice Cloning and Fake Audio Detection/data/raw/commonvoice

Directory exists:
True

MP3 files:
Count: 38581
../data/raw/commonvoice/test/spontaneous-speech-ady-67085.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67086.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67087.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67093.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67157.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-68072.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67209.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67208.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67220.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-68073.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67156.mp3
../data/raw/commonvoice/test/spontaneous-speech-ady-67154.mp3
../data/raw/commonvoice/test/s

In [6]:

from pathlib import Path
# Path makes it easier and safer to work with file/folder paths
# instead of manually joining strings with "/".


from TTS.api import TTS
# Imports Coqui TTS, which provides access to pretrained
# text-to-speech models such as XTTS-v2.


# ============================================================
# CONFIGURATION
# ============================================================

TIMIT_RAW_DIR = Path("../data/raw/timit")
# Location of the ORIGINAL TIMIT dataset.
# This directory contains the original audio files and
# their corresponding .TXT transcript files.


TIMIT_DIR = Path(
    "../data/processed/timit"
)
# Location of the PROCESSED TIMIT audio files.
# These files were previously cleaned/resampled during
# the preprocessing step.


TIMIT_PROCESSED_DIR = Path("../data/processed/timit")
# Same processed TIMIT directory.
# This variable is used specifically when calculating
# the relative path to the original TIMIT transcript.


OUTPUT_DIR = Path(
    "../data/processed/generated"
)
# Directory where XTTS-v2 generated/fake audio will be saved.


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)
# Creates the output directory if it does not already exist.
#
# parents=True:
# Creates any missing parent directories as well.
#
# exist_ok=True:
# Does not produce an error if the directory already exists.


# ============================================================
# LOAD MODEL
# ============================================================

print("Loading XTTS-v2...")
# Lets us know that the pretrained TTS model is being loaded.


tts = TTS(
    "tts_models/multilingual/multi-dataset/xtts_v2"
)
# Loads the pretrained XTTS-v2 text-to-speech model.
#
# XTTS-v2 can:
# 1. Take a text sentence.
# 2. Take a reference speaker recording.
# 3. Generate new speech that attempts to sound like
#    the reference speaker saying the provided text.
#
# This is the step that creates our SYNTHETIC/FAKE audio.


print("XTTS-v2 loaded.")
# Confirms that the model finished loading successfully.


# ============================================================
# FIND ALL TIMIT AUDIO FILES
# ============================================================

wav_files = list(
    TIMIT_DIR.rglob("*.wav")
)
# Recursively searches all folders under TIMIT_DIR
# for lowercase .wav files.
#
# rglob() means "recursive glob".
#
# For example, it can find:
#
# ../data/processed/timit/TRAIN/DR1/MMAB0/SA1.wav
# ../data/processed/timit/TRAIN/DR1/MMAB0/SA2.wav


wav_files += list(
    TIMIT_DIR.rglob("*.WAV")
)
# Also searches for uppercase .WAV files.
#
# This is useful because the original TIMIT dataset commonly
# contains filenames/extensions such as .WAV.


# Remove duplicates
wav_files = list(set(wav_files))
# Converts the list to a set and back to a list.
#
# A set automatically removes duplicate paths.
# This protects us from processing the same audio file twice.


print(
    f"Found {len(wav_files)} TIMIT audio files."
)
# Prints the total number of processed TIMIT audio files found.


# ============================================================
# GET SPEAKER ID
# ============================================================

def get_speaker_id(wav_file):
    # Receives the path of a TIMIT audio file.

    return wav_file.parent.name
    # The parent folder represents the speaker ID.
    #
    # Example:
    #
    # .../TRAIN/DR1/MMAB0/SA2.WAV.wav
    #
    # parent folder = MMAB0
    #
    # Therefore:
    # speaker_id = "MMAB0"


# ============================================================
# GET TRANSCRIPT
# ============================================================

def get_transcript(
    processed_wav,
    raw_timit_dir
):
    # Finds the original TIMIT transcript associated
    # with a processed audio file.
    #
    # processed_wav = processed TIMIT audio file
    # raw_timit_dir = original TIMIT dataset directory


    # Example processed file:
    #
    # processed/timit/TRAIN/DR1/MMAB0/SA2.WAV.wav
    #
    # We need:
    #
    # raw/timit/TRAIN/DR1/MMAB0/SA2.TXT


    relative_path = (
        processed_wav
        .relative_to(TIMIT_PROCESSED_DIR)
    )
    # Converts the absolute/long processed path into a path
    # relative to the processed TIMIT directory.
    #
    # Example:
    #
    # TRAIN/DR1/MMAB0/SA2.WAV.wav
    #
    # This allows us to reconstruct the corresponding
    # location inside the raw TIMIT dataset.


    # Remove .wav
    filename = relative_path.name
    # Extracts only the filename.
    #
    # Example:
    #
    # TRAIN/DR1/MMAB0/SA2.WAV.wav
    #
    # becomes:
    #
    # SA2.WAV.wav


    original_name = Path(
        filename
    ).stem
    # Removes the final extension.
    #
    # SA2.WAV.wav
    #
    # becomes:
    #
    # SA2.WAV


    # Remove original .WAV
    if original_name.upper().endswith(".WAV"):
        # Checks whether the remaining filename ends with
        # the original TIMIT .WAV extension.
        #
        # .upper() makes the check case-insensitive.

        original_name = original_name[:-4]
        # Removes the final ".WAV".
        #
        # SA2.WAV
        #
        # becomes:
        #
        # SA2


    # Construct raw TXT path
    txt_path = (
        raw_timit_dir
        / relative_path.parent
        / f"{original_name}.TXT"
    )
    # Builds the path to the original TIMIT transcript.
    #
    # Example:
    #
    # ../data/raw/timit/
    #     TRAIN/DR1/MMAB0/
    #     SA2.TXT
    #
    # TIMIT stores the transcript separately from the audio.


    if not txt_path.exists():
        # If the uppercase .TXT file does not exist...

        txt_path = (
            raw_timit_dir
            / relative_path.parent
            / f"{original_name}.txt"
        )
        # Try lowercase .txt instead.
        #
        # This makes the code more tolerant of different
        # filename capitalization.


    if not txt_path.exists():
        # If neither transcript filename exists...

        print(
            f"Transcript not found: {txt_path}"
        )
        # Tell us which transcript could not be found.

        return None
        # Return None so the calling code knows that
        # this audio file cannot be processed.


    with open(
        txt_path,
        "r",
        encoding="utf-8"
    ) as f:
        # Opens the transcript file in read mode.
        #
        # encoding="utf-8" ensures text is read correctly.

        line = f.readline().strip()
        # Reads the first line of the transcript
        # and removes leading/trailing whitespace.


    if not line:
        # Checks whether the transcript is empty.

        return None
        # No usable text means there is nothing to synthesize.


    parts = line.split()
    # Splits the transcript line into individual pieces.
    #
    # TIMIT transcript lines generally contain:
    #
    # start_time end_time transcript
    #
    # Example:
    #
    # 0 46797 She had your dark suit in greasy wash water


    if len(parts) < 3:
        # Makes sure the line contains at least:
        #
        # 1. start time
        # 2. end time
        # 3. transcript text

        return None
        # If the format is invalid, skip this sample.


    return " ".join(
        parts[2:]
    )
    # Removes the first two fields:
    #
    # start_time
    # end_time
    #
    # and returns only the actual spoken sentence.
    #
    # Example:
    #
    # ["0", "46797", "She", "had", "your", "dark", "suit"]
    #
    # becomes:
    #
    # "She had your dark suit"


# ============================================================
# GROUP AUDIO BY SPEAKER
# ============================================================

speakers = {}
# Creates an empty dictionary.
#
# The dictionary will eventually look something like:
#
# {
#     "MMAB0": [audio1, audio2, audio3, ...],
#     "MDAB0": [audio1, audio2, audio3, ...]
# }
#
# This allows us to process each speaker separately.


for wav_file in wav_files:
    # Loops through every TIMIT audio file.

    speaker = get_speaker_id(
        wav_file
    )
    # Extracts the speaker ID from the parent folder.


    if speaker not in speakers:
        # If this is the first audio file we've seen
        # for this speaker...

        speakers[speaker] = []
        # Create an empty list for that speaker.


    speakers[speaker].append(
        wav_file
    )
    # Adds this audio file to that speaker's list.


print(
    f"Found {len(speakers)} speakers."
)
# Prints the total number of unique speakers.


# ============================================================
# GENERATE ALL AUDIO
# ============================================================

total_generated = 0
# Counts successfully generated audio files.


total_failed = 0
# Counts files that could not be generated.


for speaker, files in speakers.items():
    # Processes one speaker at a time.
    #
    # speaker = speaker ID
    # files = all recordings belonging to that speaker.


    print()
    print("=" * 60)
    print(f"SPEAKER: {speaker}")
    print(f"Utterances: {len(files)}")
    print("=" * 60)
    # Prints information about the current speaker.
    #
    # An utterance is one spoken recording/sentence.


    # --------------------------------------------------------
    # Need at least two files:
    #
    # One = reference voice
    # Others = text to synthesize
    # --------------------------------------------------------

    if len(files) < 2:
        # We need at least two recordings.
        #
        # One recording will provide the speaker's voice.
        # Another recording provides a transcript that
        # XTTS-v2 will synthesize.


        print(
            f"Skipping {speaker}: "
            f"not enough audio files."
        )
        # Explains why this speaker is being skipped.


        continue
        # Move to the next speaker.


    # --------------------------------------------------------
    # Sort files so results are reproducible
    # --------------------------------------------------------

    files = sorted(files)
    # Sorts the audio paths alphabetically.
    #
    # This ensures that files[0] is consistently selected
    # as the reference recording each time the script runs.


    # --------------------------------------------------------
    # Use first recording as reference
    # --------------------------------------------------------

    reference_audio = files[0]
    # Selects the first recording as the reference voice.
    #
    # XTTS-v2 analyzes this recording to imitate the
    # speaker's voice characteristics.


    print(
        f"Reference audio: "
        f"{reference_audio.name}"
    )
    # Shows which recording is being used as the
    # speaker reference.


    # --------------------------------------------------------
    # Generate speech using every OTHER recording
    # --------------------------------------------------------

    for source_audio in files[1:]:
        # Loops through every recording AFTER the reference.
        #
        # files[1:] means:
        #
        # skip files[0]
        # process files[1], files[2], files[3], ...
        #
        # The transcript from each source recording becomes
        # the text that XTTS-v2 synthesizes.


        text = get_transcript(
            source_audio, 
            TIMIT_RAW_DIR    
        )
        # Finds the transcript corresponding to this
        # source audio file.


        if text is None:
            # If a transcript could not be found...

            print(
                f"Skipping {source_audio.name}: "
                f"transcript not found."
            )
            # Explain why this file is being skipped.


            total_failed += 1
            # Increase the failure counter.

            continue
            # Move to the next source recording.


        # ----------------------------------------------------
        # Create output filename
        # ----------------------------------------------------

        output_filename = (
            f"{speaker}_"
            f"{source_audio.stem}_"
            f"fake.wav"
        )
        # Creates a filename identifying:
        #
        # speaker
        # source recording
        # fake/synthetic audio
        #
        # Example:
        #
        # MMAB0_SA2.WAV_fake.wav


        output_file = (
            OUTPUT_DIR /
            output_filename
        )
        # Combines the output directory with the filename.
        #
        # Example:
        #
        # ../data/processed/generated/MMAB0_SA2.WAV_fake.wav


        # ----------------------------------------------------
        # Don't regenerate existing files
        # ----------------------------------------------------

        if output_file.exists():
            # Checks whether this synthetic audio file
            # was already generated.

            print(
                f"Already exists: "
                f"{output_file.name}"
            )
            # Lets us know that generation is being skipped.


            total_generated += 1
            # Counts the existing file as already generated.


            continue
            # Move to the next source recording.
            #
            # This prevents wasting time regenerating
            # files that already exist.


        print()
        print(
            f"Generating: "
            f"{output_file.name}"
        )
        # Shows which fake audio file is currently
        # being generated.


        print(
            f"Reference: "
            f"{reference_audio.name}"
        )
        # Shows the audio providing the speaker's voice.


        print(
            f"Source: "
            f"{source_audio.name}"
        )
        # Shows which original TIMIT recording supplied
        # the transcript.


        print(
            f"Text: {text}"
        )
        # Prints the exact sentence that XTTS-v2
        # will synthesize.


        # ----------------------------------------------------
        # XTTS GENERATION
        # ----------------------------------------------------

        try:
            # Starts an exception-handling block.
            #
            # If XTTS encounters an error for one file,
            # the entire program does not have to stop.


            tts.tts_to_file(
                # Generates speech and saves it directly
                # to an audio file.

                text=text,
                # Text that XTTS-v2 should speak.


                speaker_wav=str(
                    reference_audio
                ),
                # Reference recording used to imitate
                # the target speaker's voice.
                #
                # str() converts the Path object into
                # a normal string path.


                language="en",
                # Tells XTTS-v2 that the generated speech
                # should be English.


                file_path=str(
                    output_file
                )
                # Location where the generated synthetic
                # audio will be saved.
            )


            total_generated += 1
            # Increase the successful generation counter.


            print(
                "✓ Generation complete."
            )
            # Confirms that this audio file was successfully
            # generated.


        except Exception as e:
            # If XTTS-v2 encounters an error, execution
            # enters this block instead of crashing.


            total_failed += 1
            # Increase the failure counter.


            print(
                f"✗ Generation failed: {e}"
            )
            # Prints the actual error message so we can
            # troubleshoot the failed file.


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 60)
print("VOICE CLONING COMPLETE")
print("=" * 60)
# Prints a clear completion message after all speakers
# and recordings have been processed.


print(
    f"Generated: {total_generated}"
)
# Shows how many synthetic files were generated
# or already existed.


print(
    f"Failed: {total_failed}"
)
# Shows how many files could not be processed.


print(
    f"Output directory:"
)
# Labels the location of the generated audio files.


print(
    OUTPUT_DIR.resolve()
)
# Converts the relative output path into an absolute path
# and prints it so you know exactly where the generated
# fake audio files are located.
```


Loading XTTS-v2...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
XTTS-v2 loaded.
Found 5670 TIMIT audio files.
Found 567 speakers.

SPEAKER: MWCH0
Utterances: 10
Reference audio: SA1.WAV.wav

Generating: MWCH0_SA2.WAV_fake.wav
Reference: SA1.WAV.wav
Source: SA2.WAV.wav
Text: Don't ask me to carry an oily rag like that.
 > Text splitted to sentences.
["Don't ask me to carry an oily rag like that."]


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 > Processing time: 2.1686270236968994
 > Real-time factor: 0.68155966180896
✓ Generation complete.

Generating: MWCH0_SI1622.WAV_fake.wav
Reference: SA1.WAV.wav
Source: SI1622.WAV.wav
Text: She knew she was feeling afraid and inwardly laughed at herself.
 > Text splitted to sentences.
['She knew she was feeling afraid and inwardly laughed at herself.']
 > Processing time: 2.7877981662750244
 > Real-time factor: 0.6419272093396439
✓ Generation complete.

Generating: MWCH0_SI1895.WAV_fake.wav
Reference: SA1.WAV.wav
Source: SI1895.WAV.wav
Text: It might hurt you, though.
 > Text splitted to sentences.
['It might hurt you, though.']
 > Processing time: 0.9783520698547363
 > Real-time factor: 0.6059736837162061
✓ Generation complete.

Generating: MWCH0_SI2252.WAV_fake.wav
Reference: SA1.WAV.wav
Source: SI2252.WAV.wav
Text: Half slyly he enjoyed seeing her stoop to lift the things.
 > Text splitted to sentences.
['Half slyly he enjoyed seeing her stoop to lift the things.']
 > Processing ti

KeyboardInterrupt: 